# Model Evaluation Report

All numbers below are read from the served model's `metrics.json`, written by
`scripts/train.py`. The test set is **whole drive profiles the model never saw**:
the data is sampled at 2 Hz, so a random row split would put near-identical rows in
both train and test and inflate every metric.

Read the results against the **always-Nominal baseline**. About 91% of rows are
Nominal, so accuracy alone says very little.

In [ ]:
import json
from pathlib import Path

import pandas as pd

models = Path('../models')
version = (models / 'latest.txt').read_text().strip()
m = json.loads((models / version / 'metrics.json').read_text())
print('Model version:', version)
print('Split:', m['benchmark_split'], '| test drives:', m['test_groups'], '| test rows:', m['test_rows'])

## 1. Headline metrics vs. baseline

In [ ]:
base = m['baseline_always_nominal']
pd.DataFrame({
    'Model': [m['test_accuracy'], m['test_f1_macro'], m['cv_f1_macro_mean']],
    'Always Nominal': [base['test_accuracy'], base['test_f1_macro'], None],
}, index=['Accuracy (held-out drives)', 'Macro F1 (held-out drives)', 'Macro F1 (drive-level CV mean)']).round(3)

## 2. Per-class performance

In [ ]:
per_class = pd.DataFrame(m['per_class_report']).T[['precision', 'recall', 'f1-score', 'support']]
per_class['drives_with_class'] = [m['profiles_per_class'][str(i)] for i in range(len(per_class))]
per_class.round(3)

## 3. Confusion matrix (held-out drives)

In [ ]:
labels = m['confusion_matrix_labels']
pd.DataFrame(m['confusion_matrix'], index=[f'actual: {l}' for l in labels], columns=labels)

In [ ]:
print('Not evaluable:', m.get('not_evaluable_classes'))
print('CV macro F1: %.3f ± %.3f' % (m['cv_f1_macro_mean'], m['cv_f1_macro_std']))
for note in m['leakage_notes']:
    print('-', note)

## Limitations

- **The labels are proxies, not observed failures.** The PMSM dataset has no fault
  annotations. A "fault" means a hidden thermal channel (stator winding, tooth, yoke
  or magnet temperature) is in its top few percent. The model therefore gives early
  warning of thermal stress from electrical and control signals; it does not detect
  component failures.
- **Some classes can't be evaluated.** A class that occurs in fewer than two drives
  can't be both learned and tested on an unseen drive (see `not_evaluable_classes`),
  and it is left out of macro F1.
- **Scores vary from drive to drive.** With only about 13 test drives, scores change
  noticeably between splits. Treat the drive-level CV standard deviation as the error bar.